# 09 — Capteurs : import, analyse HRV et sauvegarde en base

Ce notebook illustre le workflow complet pour chaque source de données supportée :

1. **Importer** les données brutes (Polar, HRV4Training, Apple Watch, Garmin)
2. **Nettoyer** les intervalles RR avec `remove_outliers()`
3. **Calculer** les features HRV (domaine temporel, spectral)
4. **Scorer** la session via les protocoles cardiolab
5. **Sauvegarder** en base de données (optionnel — nécessite PostgreSQL)

> **Protocole de mesure recommandé** : le matin, allongé, 5 minutes immobile, respiration naturelle.  
> Voir `cardiolab/docs/sensors/` pour les instructions spécifiques à chaque capteur.


In [ ]:
from pathlib import Path

import cardiolab

DATASETS_DIR = Path(cardiolab.__file__).parent / "datasets"
RAW_DIR = DATASETS_DIR / "raw"

print(f"cardiolab {cardiolab.__version__}")
print(f"datasets : {DATASETS_DIR}")

---
## 1. Polar H10 — intervalles RR bruts

**Protocole** :
- Ceinture thoracique Polar H10 + app Polar Sensor Logger
- 5 minutes allongé, le matin au réveil
- Fichier exporté : `.txt` avec horodatage + intervalles RR en ms

**Avantage** : accès complet aux RR → RMSSD, SDNN, HF, pNN50, DFA α1.  
**Limite** : nécessite le capteur physique.


In [ ]:
from cardiolab.features.frequency_domain import compute_frequency_domain
from cardiolab.features.time_domain import compute_time_domain
from cardiolab.sensors_tools import parse_rr_file

polar_file = RAW_DIR / "polar" / "session.txt"  # remplacer par votre fichier

if polar_file.exists():
    rr_polar = parse_rr_file(polar_file).remove_outliers()
    print(f"Polar — {len(rr_polar)} intervals, mean HR: {rr_polar.mean_hr:.0f} bpm")

    td_polar = compute_time_domain(rr_polar)
    print(f"RMSSD : {td_polar.rmssd:.1f} ms | SDNN : {td_polar.sdnn:.1f} ms")

    fd_polar = compute_frequency_domain(rr_polar)
    print(
        f"LF : {fd_polar.lf:.2f} ms² | HF : {fd_polar.hf:.2f} ms² | LF/HF : {fd_polar.lf_hf_ratio:.2f}"
    )
else:
    import numpy as np

    from cardiolab.signals.rr import RRSeries

    rng = np.random.default_rng(42)
    rr_polar = RRSeries(800 + rng.normal(0, 30, 300))
    rr_polar = rr_polar.remove_outliers()
    td_polar = compute_time_domain(rr_polar)
    print("[démo] Données synthétiques Polar")
    print(f"RMSSD : {td_polar.rmssd:.1f} ms | SDNN : {td_polar.sdnn:.1f} ms")

---
## 2. HRV4Training — CSV avec intervalles RR

**Protocole** :
- App HRV4Training (iOS/Android) — mesure par caméra ou ceinture
- Exporter via Settings → Export Data → CSV
- Le CSV contient une ligne par jour avec date, rMSSD, FC repos, et les intervalles RR en semicolons

**Avantage** : pas de capteur physique requis, historique multi-jours en un fichier.  
**Limite** : résolution caméra moins précise que ceinture thoracique.


In [ ]:
import os
import tempfile

from cardiolab.sensors_tools import parse_hrv4training_csv, to_rrseries

hrv4t_file = RAW_DIR / "hrv4training" / "export.csv"  # remplacer par votre fichier

# Données de démonstration embarquées
_HRV4T_DEMO = (
    "date,rMSSD,resting HR,RR intervals\n"
    "2026-01-10,55.2,56,800;810;795;820;805;790;815;800;810;795\n"
    "2026-01-11,48.1,59,850;840;830;845;820;835;850;840\n"
    "2026-01-12,61.3,54,780;790;785;800;795;780;790;795;800\n"
)

if hrv4t_file.exists():
    records = parse_hrv4training_csv(hrv4t_file)
else:
    fd, tmp_path = tempfile.mkstemp(suffix=".csv")
    try:
        os.write(fd, _HRV4T_DEMO.encode())
        os.close(fd)
        records = parse_hrv4training_csv(tmp_path)
    finally:
        os.unlink(tmp_path)
    print("[démo] Données synthétiques HRV4Training")

print(f"{len(records)} sessions importées")
for rec in records:
    rr = to_rrseries(rec)
    td = compute_time_domain(rr)
    print(f"  {rec['date']} — rMSSD CSV: {rec['rmssd']} | calculé: {td.rmssd:.1f} ms")

---
## 3. Apple Watch — export Apple Health (XML)

**Protocole** :
- Sur iPhone : Santé → Profil → Exporter les données de santé → `export.zip`
- Décompresser → `apple_health_export/export.xml`

> ⚠️ **Limite importante** : l'export Apple Health ne contient **pas les intervalles RR bruts** —  
> seulement le **SDNN** pré-calculé par watchOS. Le champ `rmssd` est toujours `None`.  
> Pour une analyse HRV complète, utiliser Polar H10 ou Garmin avec ceinture.


In [ ]:
import os
import tempfile

from cardiolab.sensors_tools import extract_hrv_samples

apple_file = RAW_DIR / "apple_health" / "export.xml"  # remplacer par votre fichier

_APPLE_DEMO_XML = """<?xml version=\"1.0\" encoding=\"UTF-8\"?>
<HealthData locale=\"fr_FR\">
  <Record type=\"HKQuantityTypeIdentifierHeartRateVariabilitySDNN\"
          sourceName=\"Apple Watch de Julien\" unit=\"ms\" value=\"45.2\"
          startDate=\"2026-01-10 07:30:00 +0000\" endDate=\"2026-01-10 07:35:00 +0000\"/>
  <Record type=\"HKQuantityTypeIdentifierHeartRateVariabilitySDNN\"
          sourceName=\"Apple Watch de Julien\" unit=\"ms\" value=\"38.7\"
          startDate=\"2026-01-11 07:28:00 +0000\" endDate=\"2026-01-11 07:33:00 +0000\"/>
  <Record type=\"HKQuantityTypeIdentifierRestingHeartRate\"
          sourceName=\"Apple Watch de Julien\" unit=\"count/min\" value=\"56\"
          startDate=\"2026-01-10 07:35:00 +0000\" endDate=\"2026-01-10 07:35:00 +0000\"/>
</HealthData>"""

if apple_file.exists():
    samples = extract_hrv_samples(apple_file)
else:
    fd, tmp_path = tempfile.mkstemp(suffix=".xml")
    try:
        os.write(fd, _APPLE_DEMO_XML.encode())
        os.close(fd)
        samples = extract_hrv_samples(tmp_path)
    finally:
        os.unlink(tmp_path)
    print("[démo] Données synthétiques Apple Health")

print(f"{len(samples)} jours importés")
for s in samples:
    print(
        f"  {s['date']} — SDNN: {s['sdnn']:.1f} ms | RMSSD: {s['rmssd']} (non disponible)"
    )

---
## 4. Garmin — FIT (intervalles RR bruts)

**Protocole** :
- Ceinture Garmin HRM-Pro ou HRM-Dual (ou Polar H10 via ANT+)
- Créer une activité "Méditation" ou "Repos" de 5 min sur la montre
- Exporter depuis Garmin Connect : Activité → ⚙️ → Exporter vers FIT
- Installer le support FIT : `pip install cardiolab[garmin]`

**Avantage** : RR bruts de qualité ECG si ceinture thoracique.  
**Limite** : nécessite `fitparse` + ceinture pour avoir les messages HRV.


In [ ]:
try:
    from cardiolab.sensors_tools import parse_garmin_fit

    garmin_fit = RAW_DIR / "garmin" / "activity.fit"  # remplacer par votre fichier

    if garmin_fit.exists():
        rr_garmin = parse_garmin_fit(garmin_fit).remove_outliers()
        td_garmin = compute_time_domain(rr_garmin)
        print(f"Garmin FIT — {len(rr_garmin)} intervals")
        print(f"RMSSD : {td_garmin.rmssd:.1f} ms | SDNN : {td_garmin.sdnn:.1f} ms")
    else:
        print(
            "[démo] Pas de fichier FIT — utiliser un fichier Garmin réel pour tester."
        )
        print("Déposer le fichier dans : datasets/raw/garmin/activity.fit")

except ImportError as e:
    print(f"fitparse non installé : {e}")
    print("Installer avec : pip install cardiolab[garmin]")

---
## 5. Garmin — CSV beat-to-beat (Kubios / outils tiers)

Certains outils tiers (Kubios HRV, HRVanalysis) permettent d'exporter les intervalles RR
depuis un fichier FIT vers un CSV beat-to-beat. Ce format est directement compatible avec
`parse_garmin_csv()` — aucune dépendance supplémentaire requise.


In [ ]:
import os
import tempfile

from cardiolab.sensors_tools import parse_garmin_csv

_GARMIN_CSV_DEMO = (
    "Timestamp,RR Interval (ms),HR (bpm)\n"
    "2026-01-10T07:30:00,812,74\n"
    "2026-01-10T07:30:01,800,75\n"
    "2026-01-10T07:30:02,795,75\n"
    "2026-01-10T07:30:02,820,73\n"
    "2026-01-10T07:30:03,810,74\n"
)

garmin_csv = RAW_DIR / "garmin" / "rr_export.csv"

if garmin_csv.exists():
    rr_gcsv = parse_garmin_csv(garmin_csv).remove_outliers()
else:
    fd, tmp_path = tempfile.mkstemp(suffix=".csv")
    try:
        os.write(fd, _GARMIN_CSV_DEMO.encode())
        os.close(fd)
        rr_gcsv = parse_garmin_csv(tmp_path).remove_outliers()
    finally:
        os.unlink(tmp_path)
    print("[démo] Données synthétiques Garmin CSV")

td_gcsv = compute_time_domain(rr_gcsv)
print(f"Garmin CSV — {len(rr_gcsv)} intervals")
print(f"RMSSD : {td_gcsv.rmssd:.1f} ms | SDNN : {td_gcsv.sdnn:.1f} ms")

---
## 6. Comparaison des sources disponibles

Ce tableau récapitule ce que chaque source permet de calculer.


In [ ]:
comparison = [
    {
        "Source": "Polar H10",
        "Raw RR": "✅",
        "RMSSD": "✅",
        "SDNN": "✅",
        "Resting HR": "❌",
        "Extra dep.": "—",
    },
    {
        "Source": "HRV4Training",
        "Raw RR": "✅",
        "RMSSD": "✅",
        "SDNN": "✅",
        "Resting HR": "✅",
        "Extra dep.": "—",
    },
    {
        "Source": "Apple Watch (XML)",
        "Raw RR": "❌",
        "RMSSD": "❌",
        "SDNN": "✅",
        "Resting HR": "✅",
        "Extra dep.": "—",
    },
    {
        "Source": "Garmin FIT",
        "Raw RR": "✅",
        "RMSSD": "✅",
        "SDNN": "✅",
        "Resting HR": "❌",
        "Extra dep.": "cardiolab[garmin]",
    },
    {
        "Source": "Garmin CSV (RR)",
        "Raw RR": "✅",
        "RMSSD": "✅",
        "SDNN": "✅",
        "Resting HR": "❌",
        "Extra dep.": "—",
    },
]

header = ["Source", "Raw RR", "RMSSD", "SDNN", "Resting HR", "Extra dep."]
widths = [max(len(r[h]) for r in comparison + [{h: h}]) for h in header]
fmt = "  ".join(f"{{:<{w}}}" for w in widths)

print(fmt.format(*header))
print("  ".join("-" * w for w in widths))
for row in comparison:
    print(fmt.format(*[row[h] for h in header]))

---
## 7. Scoring HRV et sauvegarde en base

Une fois les intervalles RR importés depuis n'importe quel capteur, le pipeline
de scoring et de sauvegarde est identique — le capteur source n'a pas d'impact
sur le reste du workflow.

> **Prérequis** : PostgreSQL configuré via `.env` (voir `cardiolab/docs/database/`).


In [ ]:
from cardiolab.protocols.resting import RestingProtocol

# Utiliser les données Polar importées (ou toute autre source)
protocol = RestingProtocol(rr_polar)
result = protocol.run()

print(f"Score readiness : {result.score:.1f} / 100")
print(f"RMSSD          : {result.rmssd:.1f} ms")
print(f"SDNN           : {result.sdnn:.1f} ms")
print(f"HR moyen       : {result.mean_hr:.0f} bpm")

In [ ]:
# Sauvegarde en base (optionnel — nécessite PostgreSQL)
try:
    import os

    from cardiolab.storage.database import DatabaseSession
    from dotenv import load_dotenv

    load_dotenv()
    db_url = os.getenv("DATABASE_URL", "")

    if db_url:
        with DatabaseSession(db_url) as db:
            session_id = db.save_resting_session(result)
            print(f"Session sauvegardée — ID : {session_id}")
    else:
        print(
            "[skip] DATABASE_URL non défini — configurer .env pour activer la sauvegarde."
        )

except ImportError:
    print("[skip] psycopg2 non installé : pip install cardiolab[database]")

---
## Ressources / Resources

- `cardiolab/docs/sensors/README.md` — tableau de compatibilité complet
- `cardiolab/docs/sensors/polar.md` — protocole Polar H10
- `cardiolab/docs/sensors/hrv4training.md` — protocole HRV4Training
- `cardiolab/docs/sensors/apple_health.md` — export Apple Watch
- `cardiolab/docs/sensors/garmin.md` — export Garmin FIT/CSV
- `notebooks/02_import_pipeline.ipynb` — pipeline d'import complet
- `notebooks/08_full_pipeline.ipynb` — pipeline quotidien complet
